In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv('merged_file.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['latitude', 'longitude', 'date']).reset_index(drop=True)

print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Loaded: (94417, 12)
Columns: ['latitude', 'longitude', 'date', 'fire_intensity', 'confidence', 'temp_2m', 'soil_moisture', 'dewpoint_2m', 'surface_pressure', 'wind_u', 'wind_v', 'total_precipitation']


,latitude,longitude,date,fire_intensity,confidence,temp_2m,soil_moisture,dewpoint_2m,surface_pressure,wind_u,wind_v,total_precipitation
0,8.25338,77.46686,2022-03-05,1.24,n,302.28070,0.124702,295.377385,99565.9540,-0.345467,-0.035397,0.000259
1,8.25538,77.47021,2022-03-07,2.63,n,302.44996,0.134691,295.584375,99509.6525,-1.448208,-1.045975,0.000046
2,8.25599,77.46732,2022-03-07,2.64,n,302.44996,0.134691,295.584375,99509.6525,-1.448208,-1.045975,0.000046


In [3]:
# tempreature conversion
df['temp_c']     = df['temp_2m']     - 273.15
df['dewpoint_c'] = df['dewpoint_2m'] - 273.15

# VPD
es = 0.6108 * np.exp((17.27 * df['temp_c'])     / (df['temp_c']     + 237.3))
ea = 0.6108 * np.exp((17.27 * df['dewpoint_c']) / (df['dewpoint_c'] + 237.3))
df['vpd'] = es - ea

# Wind speed
df['wind_speed'] = np.sqrt(df['wind_u']**2 + df['wind_v']**2)

# Soil moisture rolling per location
for window, col in [(7,'soil_moisture_7d'),(14,'soil_moisture_14d'),(30,'soil_moisture_30d')]:
    df[col] = (df.groupby(['latitude','longitude'])['soil_moisture']
                 .rolling(window=window, min_periods=1).mean()
                 .reset_index(level=[0,1], drop=True).sort_index())
    
# VPD rolling per location
for window, col in [(7,'vpd_7d'),(14,'vpd_14d')]:
    df[col] = (df.groupby(['latitude','longitude'])['vpd']
                 .rolling(window=window, min_periods=1).mean()
                 .reset_index(level=[0,1], drop=True).sort_index())
    
# Month and season
df['month'] = df['date'].dt.month
season_map = {12:1,1:1,2:1, 3:2,4:2,5:2, 6:4,7:4,8:4, 9:3,10:3,11:3}
df['season_enc'] = df['month'].map(season_map)

# Confidence encoding
df['confidence_enc'] = df['confidence'].map({'l':0,'n':1,'h':2})

# Re-sort by date for modeling
df = df.sort_values('date').reset_index(drop=True)

print("Shape after feature engineering:", df.shape)
print("Nulls:", df.isnull().sum().sum())


Shape after feature engineering: (94417, 24)
Nulls: 0


In [4]:
# Save feature engineered dataset
df.to_csv('wildfire_features_engineered.csv', index=False)

# Confirm
confirm = pd.read_csv('wildfire_features_engineered.csv')
print(f"Saved : {confirm.shape}")
print(f"Cols  : {confirm.columns.tolist()}")

Saved : (94417, 24)
Cols  : ['latitude', 'longitude', 'date', 'fire_intensity', 'confidence', 'temp_2m', 'soil_moisture', 'dewpoint_2m', 'surface_pressure', 'wind_u', 'wind_v', 'total_precipitation', 'temp_c', 'dewpoint_c', 'vpd', 'wind_speed', 'soil_moisture_7d', 'soil_moisture_14d', 'soil_moisture_30d', 'vpd_7d', 'vpd_14d', 'month', 'season_enc', 'confidence_enc']
